#### Objetivo

- Esse notebook consiste em concatenar os dados dos eventos de todas as partidas das competições entre todas as suas temporadas. 

    A ideia é de analisar principalmente a disponibilidade dos dados de tracking e entender qual/quais temporadas sugerem estar mais consistentes de serem utilizadas.

In [ ]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [ ]:
competitions = [competition for competition in os.listdir(str(Path().resolve().parent.parent / "data" / "events"))]
competitions

In [ ]:
competitions_seasons = {
    competitions[id]: [season for season in os.listdir(str(Path().resolve().parent.parent / "data" / "events" / competitions[id]))]
    for id in range(len(competitions))
}
competitions_seasons

In [ ]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [ ]:
def get_competition_seasons_parquet_file_paths(competition, seasons):

    competition_seasons_parquet_file_paths = []

    for season in seasons:

        events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / competition / season)

        competition_seasons_parquet_file_paths.extend(get_season_events_parquet_file_paths(events_competition_season_folder_path))
    
    return competition_seasons_parquet_file_paths

In [ ]:
competitions_seasons_events_parquet_file_paths = []

for competition, seasons in competitions_seasons.items():

    competitions_seasons_events_parquet_file_paths.extend(get_competition_seasons_parquet_file_paths(competition, seasons))

#competitions_seasons_events_parquet_file_paths

In [ ]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("season_events_sanity")
    .getOrCreate()
)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df = spark.read.parquet(*competitions_seasons_events_parquet_file_paths)

df.show(5)

In [ ]:
df = df.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            #StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        #StructField("visibility", StringType(), True),
        #StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        #StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df = df.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df = df.select(
    'competitionId',
    'season',
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    'homeTeam',
    'details_parsed',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [ ]:
# ============================================================
# Colunas de tipo e outcome extraídas do details_parsed, por eventType
# ============================================================
# Cada eventType usa uma chave diferente dentro do details_parsed pra guardar
# o "tipo" específico do evento e o "outcome" (resultado). Quando o eventType
# não tem uma dessas chaves (ex: Clearance não tem tipo, kickoff não tem outcome),
# a coluna fica NULL.

event_type_desc_key = (
    F.when(F.col('eventType').isin('OTB', 'FIRSTKICKOFF', 'SECONDKICKOFF'), F.lit('setpieceTypeDescription'))
    .when(F.col('eventType') == 'BC', F.lit('carryTypeDescription'))
    .when(F.col('eventType').isin('CH', 'FO'), F.lit('challengeTypeDescription'))
    .when(F.col('eventType') == 'CR', F.lit('crossTypeDescription'))
    .when(F.col('eventType') == 'PA', F.lit('passTypeDescription'))
    .when(F.col('eventType') == 'SH', F.lit('shotTypeDescription'))
    .when(F.col('eventType') == 'TC', F.lit('touchTypeDescription'))
    # CL (Clearance) e RE (Rebound) não têm chave de tipo no details_parsed
)

event_outcome_desc_key = (
    F.when(F.col('eventType') == 'BC', F.lit('ballCarryOutcomeDescription'))
    .when(F.col('eventType').isin('CH', 'FO'), F.lit('challengeOutcomeTypeDescription'))
    .when(F.col('eventType') == 'CL', F.lit('clearanceOutcomeTypeDescription'))
    .when(F.col('eventType') == 'CR', F.lit('crossOutcomeTypeDescription'))
    .when(F.col('eventType') == 'PA', F.lit('passOutcomeTypeDescription'))
    .when(F.col('eventType') == 'RE', F.lit('reboundOutcomeTypeDescription'))
    .when(F.col('eventType') == 'SH', F.lit('shotOutcomeTypeDescription'))
    .when(F.col('eventType') == 'TC', F.lit('touchOutcomeTypeDescription'))
    # OTB e os kickoffs não têm chave de outcome no details_parsed
)

df_events = (
    df
    .withColumns({
        'eventSubTypeDescription': F.element_at(F.col('details_parsed'), event_type_desc_key),
        'eventOutcomeDescription': F.element_at(F.col('details_parsed'), event_outcome_desc_key)
    })
    .drop(
        'details_parsed'
    )
)

df.show()

## Sanity Check dos dados dos eventos/partidas em geral

In [ ]:
df.printSchema()

In [ ]:
print(f'Quantidade total de eventos: {df.select('id').count()}')
print(f'Quantidade de eventos distintos: {df.select('id').distinct().count()}')

print(f'Quantidade de eventos duplicados: {df.select('id').count() - df.select('id').distinct().count()}')

In [ ]:
(
    df
    .groupBy('gameId')
    .agg(F.count(F.col('id')).alias('qtd_eventos'))
    .agg(F.round(F.avg(F.col('qtd_eventos')), 2).alias('Quantidade média de eventos por partida'))
).show()

- Existem uma quantidade média de ~2500 eventos por partida dentre todas as temporadas.

In [ ]:
(
    df
    .groupBy('eventTypeDescription')
    .agg(F.count(F.col('id')).alias('Quantidade de Eventos'))
    .sort('Quantidade de Eventos', ascending=False)
    .show(truncate=False)
)

- Existem diversos tipos de eventos, principalmente ofensivos e defensivos. 
- Alguns eventos tem baixa volumetria e/ou não contemplam o objetivo das análises e modelagens, como 'First half kick off', 'Second half kick off', eventos inválidos sem descrição como 'Unknown', substituições como 'Substitution', 'Player comes off the pitch' e 'Ball hits the woodwork or corner flag and comes back into play'.

In [ ]:
(
    df
    .groupBy('period')
    .agg(F.count(F.col('id')).alias('Quantidade de Eventos'))
    .sort('Quantidade de Eventos', ascending=False)
    .show(truncate=False)
)

- Os eventos estão distribuidos em apenas 2 periodos, o que é correto visto que todas as competições são de pontos corridos e sem prorrogação.

In [ ]:
# variável que indica o time com a posse
df.groupBy('homeTeam').count().show()
df.filter(F.col('homeTeam').isNull()).groupBy('homeTeam', 'eventTypeDescription', 'eventSubTypeDescription').count().show()

## Sanity Check dos Dados de tracking dos eventos

In [ ]:
print(df.select('homePlayers').first())
print(df.select('balls').first())

In [ ]:
# Schema em Pyspark para poder parsear o json dos dados de tracking que está como string
players_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        StructField("jerseyNum", StringType(), True)      
    ])
)

balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

In [ ]:
df = df.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": from_json("awayPlayers", players_schema),

    "balls_parsed": from_json("balls", balls_schema)
})

In [ ]:
df = df.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_home":
    exists(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_away":
    exists(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    "all_balls": 
    ((size(col("balls_parsed")) > 0) & 
    forall(
        col("balls_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
}
)

In [ ]:
df.show(5)

In [ ]:
df_agg_games = (
    df.groupBy("gameId")
      .agg(
          max("competitionId").alias("competitionId"),
          max("season").alias("season"),
          round(F.mean("has_tracking_home"), 3).alias("has_tracking_home_percent"),
          round(F.mean("has_tracking_away"), 3).alias("has_tracking_away_percent"),
          round(F.mean("all_tracking_home"), 3).alias("all_tracking_home_percent"),
          round(F.mean("all_tracking_away"), 3).alias("all_tracking_away_percent"),
          round(F.mean("all_balls"), 3).alias("all_balls_percent"),
          round(F.mean("len_tracking_home"), 3).alias("tracking_home_len"),
          round(F.mean("len_tracking_away"), 3).alias("tracking_away_len"),
      )
)

df_agg_games.cache()

In [ ]:
df_agg_games.show(5)

In [ ]:
print('Quantidade de partidas sem nenhum dado de tracking do mandante:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())
print('Quantidade de partidas sem nenhum dado de tracking do adversário:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())

print('Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do mandante:', df_agg_games.filter(col('tracking_home_len') != 11).count())
print('Quantidade de partidas que possui eventos com dados de tracking incomuns (diferente de 11) do adversário:', df_agg_games.filter(col('tracking_away_len') != 11).count())

print('Quantidade de partidas que possui eventos com dados de tracking faltando dos 22 jogadores:', df_agg_games.filter((col('all_tracking_home_percent') < 1) | (col('all_tracking_away_percent') < 1)).count())

print('Quantidade de partidas que possui eventos com dados de tracking faltando da bola:', df_agg_games.filter(col('all_balls_percent') < 1).count())
print('Quantidade de partidas sem nenhum dado de tracking da bola:', df_agg_games.filter(col('all_balls_percent') == 0).count())

- Existem 52 partidas que não possuem nenhum dado de tracking para o time mandante ou adversário.
- Existem 208 partidas que possuem dados de tracking a mais ou a menos de jogadores do time mandante e 224 partidas do time adversário.
- Existem 364 partidas que não possuem dados de tracking de todos os 22 jogadores em campo.
- Todas as partidas das temporadas há dados faltando da bola. Em 52 partidas não tem nenhum dado de tracking da bola.

In [ ]:
df_agg_games.filter((col('has_tracking_home_percent') == 0) | (col('has_tracking_away_percent') == 0)).groupBy('competitionId', 'season').count().show()

- Dentre as 52 partidas com dados de tracking faltando, todas são da temporada de 2025 no Brasileirão.

In [ ]:
df_agg_games.filter((col('all_balls_percent') == 0)).groupBy('competitionId', 'season').count().show()

- As 52 partidas que não possui dados de tracking da bola também são da temporada de 2025 no Brasileirão, sugerindo ser a mesma que faltou dados de tracking dos jogadores.

In [ ]:
df_agg_games.filter((col('tracking_home_len') != 11) | (col('tracking_home_len') != 11)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

- Apesar de existir registros faltando de tracking de jogadores em alguns eventos, são poucos registros de forma geral. 
- Porém, o ano de 2025 do Brasileirão apresenta uma quantidade elevada de 144 registros com ausência de dados de tracking de algum jogador.

In [ ]:
df_agg_games.filter((col('all_tracking_home_percent') < 1) | (col('all_tracking_away_percent') < 1)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

- Pode-se notar que todas as temporadas das duas competições possuem uma certa quantidade de dados faltando para os 22 jogadores. 
- Tratando-se de Premier League, as temporadas com a menor quantidade de dados de tracking faltando seriam as de 2021-2022, 2022-2023 e 2024-2025.
- Já sobre o Brasileirão, as temporadas de 2023 e 2024 possuem uma quantidade um pouco elevada, mas seriam as mais adequadas para se trabalhar. A temporada de 2025 possui 109 eventos faltando informações de tracking, sendo que 52 delas faltam 100% dos dados de tracking.
- Para a análise de uma temporada em específico, é recomendado **começar pela de 2022-2023 da Premier League** por apresentar o menor número de registros faltantes.

In [ ]:
df_agg_games.filter(col('all_balls_percent') < 1).groupBy('competitionId', 'season', 'GameId').count().orderBy('GameId', ascending=False).show()

- Toda partida possui exatamente 1 dado faltando da bola, então será necessário imputação independente da temporada escolhida.